# Search-12 — Branch-and-bound et qualite de borne

## Quand la relaxation LP cesse de discriminer : le sac a dos 0-1

Ce carnet etudie **trois algorithmes exacts** pour le meme probleme et mesure **ce qui decide entre eux**. La reponse usuelle — « le plus d'objets, le plus dur » — est fausse pour ce probleme : le coût dépend à la fois de `n`, de la **capacite** pour la programmation dynamique, et de la **qualite de la borne** pour la recherche arborescente.

Le sac a dos 0-1 est deja present deux fois dans la serie, sous deux angles qui ne sont pas celui-ci :

| Carnet | Usage du sac a dos | Ce qu'il repond |
|---|---|---|
| `Search-09-LinearProgramming` §6 | formulation PLNE appelee a un solveur (PuLP/CBC, 5 objets) | *comment formuler* |
| `Search-11d-Descente-Sous-Budget` | substrat d'une descente locale, la borne LP servant de barriere | *comment borner un voisinage* |
| **ce carnet** | aucun solveur appele pour la methode principale, aucune recherche locale | *pourquoi la recherche arborescente marche, et quand sa borne s'effondre* |

**Navigation** : [Index de la série](../README.md) | [Version Python](Search-12-BranchAndBound-Borne-LP-Python.ipynb) | [Search-11d](Search-11d-Descente-Sous-Budget.ipynb) | [Search-12a](Search-12a-Composer-Regards.ipynb)

**Prérequis** : boucles et tableaux C#, programmation dynamique et relaxation linéaire. **Durée estimée** : 60 minutes.

**Environnement** : .NET 9+, Google.OrTools 9.15.6755 et solveur SCIP. Le jumeau Python emploie HiGHS ; les moteurs et temps diffèrent.

## Plan

1. Formaliser le probleme et **generer trois familles d'instances** (non correlees, faiblement correlees, fortement correlees) ;
2. Implementer les trois methodes exactes — **DP**, **branch-and-bound** avec borne LP, **MILP SCIP** — et verifier qu'elles concordent ;
3. Mesurer l'effet de la **capacite** (DP) et de la **famille d'instance** (B&B) ;
4. Conclure sur le choix d'algorithme, puis trois exercices.


[← Retour à la série](../README.md) · **Précédent** : [Search-11d — Descente sous budget](Search-11d-Descente-Sous-Budget.ipynb) · **Suivant** : [Search-12a — Composer des regards](Search-12a-Composer-Regards.ipynb)

In [1]:
#r "nuget:Google.OrTools,9.15.6755"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.Text;
using Google.OrTools.LinearSolver;
using Microsoft.DotNet.Interactive.Formatting;
CultureInfo.DefaultThreadCurrentCulture=CultureInfo.InvariantCulture;
var sonde=Solver.CreateSolver("SCIP") ?? throw new Exception("SCIP indisponible");
Console.WriteLine($".NET {Environment.Version}; Google.OrTools 9.15.6755; {sonde.SolverVersion()}");

Installed Packages Google.OrTools, 9.15.6755

.NET 10.0.14; Google.OrTools 9.15.6755; SCIP 10.0.0 [LP solver: SoPlex 8.0.0]


## 1. Formalisation et familles d'instances

Le **sac a dos 0-1** : etant donnes `n` objets de poids `w_i > 0` et de valeur `v_i > 0`, et une capacite `W`, choisir `x in {0,1}^n` maximisant `sum v_i x_i` sous `sum w_i x_i <= W`.

Le probleme est **NP-difficile**, mais les instances ne se ressemblent pas. On genere trois familles, qui different par la **correlation entre poids et valeur** — un facteur qui peut influencer la relaxation et la recherche :

- **non correlee** : `v_i` et `w_i` tires independamment — les valeurs et les poids sont tires independamment ;
- **faiblement correlee** : `v_i = w_i + bruit` — les valeurs restent proches des poids ;
- **fortement correlee** : `v_i = w_i + 10` — le rapport est `1 + 10/w_i` et peut varier sensiblement pour les petits poids.


In [2]:
record Instance(int[] Poids,int[] Valeurs,int Capacite);
string[] Familles={"non_correlee","faiblement_correlee","fortement_correlee"};
Instance Genere(int n,double ratio,string famille,int seed,int echelle=1){var r=new Random(seed);int[] p=Enumerable.Range(0,n).Select(_=>r.Next(1,101)).ToArray();int[] v=p.Select(x=>famille switch {"non_correlee"=>r.Next(1,101),"faiblement_correlee"=>Math.Max(1,x+r.Next(-10,11)),"fortement_correlee"=>x+10,_=>throw new ArgumentException("Famille inconnue")}).ToArray();return new Instance(p.Select(x=>x*echelle).ToArray(),v,(int)(ratio*p.Sum())*echelle);}
foreach(string f in Familles){var x=Genere(8,0.5,f,42);var rs=x.Poids.Select((p,i)=>(double)x.Valeurs[i]/p).ToArray();double moyenne=rs.Average();double ecart=Math.Sqrt(rs.Select(z=>(z-moyenne)*(z-moyenne)).Average());Console.WriteLine($"{f,-22} ratios min={rs.Min():F2}, max={rs.Max():F2}, écart-type={ecart:F3}; poids total={x.Poids.Sum()}, capacité={x.Capacite}");}

non_correlee           ratios min=0.27, max=5.13, écart-type=1.579; poids total=317, capacité=158


faiblement_correlee    ratios min=0.54, max=1.33, écart-type=0.203; poids total=317, capacité=158


fortement_correlee     ratios min=1.14, max=1.77, écart-type=0.240; poids total=317, capacité=158


### Lecture du résultat

Pour ces huit objets, les écarts-types des rapports `v/w` sont respectivement **1,579**, **0,203** et **0,240**. La forte corrélation entre poids et valeurs n'implique donc pas ici la plus petite dispersion des ratios. La relaxation trie par ratio décroissant ; son pouvoir d'élagage dépend aussi des sous-problèmes rencontrés. Le générateur C# diffère de celui de NumPy.

## 2. Trois methodes exactes

### 2.1 Programmation dynamique

`DP[j][c]` = meilleure valeur atteignable avec les `j` premiers objets et une capacite `c`. La recurrence est `DP[j][c] = max(DP[j-1][c], DP[j-1][c-w_j] + v_j)`. Le cout est `O(n * W)` : polynomial en `n` et en la valeur numérique `W`, mais **pseudo-polynomial** quand `W` est codé en binaire : sa valeur peut être exponentielle en sa longueur de représentation.


In [3]:
int DP(Instance x){var dp=new int[x.Capacite+1];for(int i=0;i<x.Poids.Length;i++)for(int c=x.Capacite;c>=x.Poids[i];c--)dp[c]=Math.Max(dp[c],dp[c-x.Poids[i]]+x.Valeurs[i]);return dp[x.Capacite];}
Console.WriteLine("DP 0-1 définie : tableau de capacité parcouru en ordre décroissant.");

DP 0-1 définie : tableau de capacité parcouru en ordre décroissant.


### Exercice 1 — reconstruire la solution

Affectez `solutionDP` à une fonction qui remplit une table complète DP, remonte les choix 0/1 et renvoie valeur et tableau. Les fixtures contrôlent dimensions, poids, valeur et optimalité.

In [4]:
Func<Instance,(int valeur,int[] choix)?> solutionDP=null; // TODO étudiant : table DP complète et remontée des choix.
if(solutionDP is null)Console.WriteLine("Exercice à compléter : reconstruire la solution DP.");
else {foreach(var x in new[]{new Instance(new[]{3,4,5},new[]{4,5,9},7),new Instance(new[]{2,3,4},new[]{4,5,7},5),Genere(12,0.5,"non_correlee",7)}){var r=solutionDP(x);if(r is null||r.Value.choix is null||r.Value.choix.Length!=x.Poids.Length||r.Value.choix.Any(z=>z!=0&&z!=1))throw new Exception("Choix invalide");int p=r.Value.choix.Select((z,i)=>z*x.Poids[i]).Sum(),v=r.Value.choix.Select((z,i)=>z*x.Valeurs[i]).Sum();if(p>x.Capacite||v!=r.Value.valeur||v!=DP(x))throw new Exception("Témoin ou optimum invalide");Console.WriteLine($"Valeur={v}, poids={p}/{x.Capacite}, choix=[{string.Join(",",r.Value.choix)}]");}}

Exercice à compléter : reconstruire la solution DP.


### 2.2 Branch-and-bound avec borne LP

La **relaxation continue** du sac a dos a une solution immediate : trier par `v/w` decroissant, prendre les objets entiers tant que ca tient, puis **fractionner** le premier objet qui ne tient plus. Sa valeur est un **majorant** de l'optimum entier — c'est une borne admissible.

Le branch-and-bound parcourt les objets dans cet ordre et, a chaque nœud, calcule cette borne sur le **reste** a remplir : si la borne n'atteint pas la meilleure solution deja trouvee, la branche entiere est elaguee. Le nombre de nœuds developpes est donc une **mesure directe de ce que la borne sait discriminer**.


In [5]:
int[] Ordre(Instance x)=>Enumerable.Range(0,x.Poids.Length).OrderByDescending(i=>(double)x.Valeurs[i]/x.Poids[i]).ThenBy(i=>i).ToArray();
double Borne(Instance x,IEnumerable<int> indices,int reste){double total=0;foreach(int j in indices){int p=x.Poids[j],v=x.Valeurs[j];if(p<=reste){total+=v;reste-=p;}else{total+=(double)v*reste/p;break;}}return total;}
(int valeur,int noeuds,bool epuise) BB(Instance x,int budget=200000){int[] ordre=Ordre(x);int meilleur=0,noeuds=0;bool epuise=false;
 void Explore(int i,int reste,int valeur){if(epuise)return;if(noeuds>=budget){epuise=true;return;}noeuds++;meilleur=Math.Max(meilleur,valeur);if(i==ordre.Length||valeur+Borne(x,ordre.Skip(i),reste)<=meilleur+1e-10)return;int j=ordre[i];if(x.Poids[j]<=reste)Explore(i+1,reste-x.Poids[j],valeur+x.Valeurs[j]);Explore(i+1,reste,valeur);}
 Explore(0,x.Capacite,0);return(meilleur,noeuds,epuise);}
Console.WriteLine("B&B DFS défini : borne fractionnaire par ratio, budget de 200 000 nœuds.");

B&B DFS défini : borne fractionnaire par ratio, budget de 200 000 nœuds.


### 2.3 MILP SCIP, juge indépendant

Google.OrTools crée un solveur SCIP et lui transmet le programme entier. Le statut optimal, la borne et le témoin sont contrôlés. Le Python utilise HiGHS ; aucun temps n’est interchangeable entre ces moteurs.

In [6]:
(int valeur,int[] choix,double borne,string version) MILP(Instance x){using var solver=Solver.CreateSolver("SCIP")??throw new Exception("SCIP absent");var z=Enumerable.Range(0,x.Poids.Length).Select(i=>solver.MakeIntVar(0,1,$"x{i}")).ToArray();var c=solver.MakeConstraint(0,x.Capacite);var obj=solver.Objective();for(int i=0;i<z.Length;i++){c.SetCoefficient(z[i],x.Poids[i]);obj.SetCoefficient(z[i],x.Valeurs[i]);}obj.SetMaximization();var status=solver.Solve();if(status!=Solver.ResultStatus.OPTIMAL)throw new Exception($"SCIP : {status}");var choix=z.Select(v=>(int)Math.Round(v.SolutionValue())).ToArray();int poids=choix.Select((a,i)=>a*x.Poids[i]).Sum(),valeur=choix.Select((a,i)=>a*x.Valeurs[i]).Sum();if(poids>x.Capacite||Math.Abs(valeur-obj.Value())>1e-6||Math.Abs(obj.BestBound()-valeur)>1e-6)throw new Exception("Témoin ou borne SCIP incohérent");return(valeur,choix,obj.BestBound(),solver.SolverVersion());}
int Glouton(Instance x){int reste=x.Capacite,total=0;foreach(int i in Ordre(x))if(x.Poids[i]<=reste){reste-=x.Poids[i];total+=x.Valeurs[i];}return total;}
Console.WriteLine("MILP SCIP et glouton définis : statut OPTIMAL, borne et témoin entier sont vérifiés.");

MILP SCIP et glouton définis : statut OPTIMAL, borne et témoin entier sont vérifiés.


## 3. Protocole de mesure

### 3.1 Concordance des trois methodes exactes

Avant toute mesure de performance, il faut etablir que les trois implementations font le **meme** travail. Sur des instances de taille modeste (ou la DP et le B&B terminent instantanement), les trois doivent rendre **exactement le meme optimum** ; le glouton, lui, est autorise a echouer — c'est ce qui le rend instructif.


In [7]:
Console.WriteLine($"{"famille",-22} {"graine",6} {"DP",6} {"SCIP",6} {"nœuds",7} {"glouton",8}");
int verifie=0;foreach(string f in Familles)foreach(int seed in new[]{42,7,99}){var x=Genere(12,0.5,f,seed);int dp=DP(x);var bb=BB(x);var m=MILP(x);int g=Glouton(x);if(bb.epuise||dp!=bb.valeur||dp!=m.valeur)throw new Exception("Désaccord des méthodes exactes");Console.WriteLine($"{f,-22} {seed,6} {dp,6} {m.valeur,6} {bb.noeuds,7} {g,8}");verifie++;}Console.WriteLine($"Accord exact sur {verifie}/9 instances ; chaque témoin SCIP a été vérifié.");

famille                graine     DP   SCIP   nœuds  glouton


non_correlee               42    375    375      20      375


non_correlee                7    475    475      24      475


non_correlee               99    564    564      26      564


faiblement_correlee        42    242    242      15      242


faiblement_correlee         7    380    380      43      350


faiblement_correlee        99    351    351      79      327


fortement_correlee         42    310    310     271      272


fortement_correlee          7    435    435      64      432


fortement_correlee         99    396    396      93      362


Accord exact sur 9/9 instances ; chaque témoin SCIP a été vérifié.


### Lecture du résultat

Les neuf instances (trois familles et trois graines) produisent le même optimum pour la DP, le B&B terminé et SCIP. Le tableau permet de comparer le glouton à cet optimum et de voir la variation des nœuds.

### 3.2 Ce qui fait exploser la DP : la capacite, pas le nombre d'objets

La recurrence est `O(n * W)`. Avec `n` fixe a 40, faire croitre `W` d'un facteur 100 doit multiplier le travail par 100 **pour la meme instance logique**. C'est le point ou la DP cesse d'etre une option, independamment du nombre d'objets.


La capacité est arrondie à l’échelle de base puis multipliée avec les poids : la relaxation fractionnaire et sa borne sont ainsi conservées exactement.

In [8]:
var capacites=new List<(int W,double dp,double bb,int noeuds,double milp,bool epuise)>();
Console.WriteLine($"{"capacité",10} {"DP (s)",12} {"B&B (s)",12} {"nœuds",8} {"SCIP (s)",12} {"budget",10}");
foreach(int e in new[]{1,10,100}){var x=Genere(40,0.5,"non_correlee",42,e);var sw=Stopwatch.StartNew();int dp=DP(x);sw.Stop();double td=sw.Elapsed.TotalSeconds;sw.Restart();var bb=BB(x);sw.Stop();double tb=sw.Elapsed.TotalSeconds;sw.Restart();var m=MILP(x);sw.Stop();double tm=sw.Elapsed.TotalSeconds;if(dp!=m.valeur||(!bb.epuise&&dp!=bb.valeur))throw new Exception("Désaccord");capacites.Add((x.Capacite,td,tb,bb.noeuds,tm,bb.epuise));Console.WriteLine($"{x.Capacite,10} {td,12:F6} {tb,12:F6} {bb.noeuds,8} {tm,12:F6} {(bb.epuise?"épuisé":"terminé"),10}");}
Console.WriteLine($"Facteur de temps DP mesuré : {capacites[^1].dp/capacites[0].dp:F2} ; capacité multipliée par 100.");

  capacité       DP (s)      B&B (s)    nœuds     SCIP (s)     budget


       859     0.000444     0.000082      185     0.007271    terminé


      8590     0.004445     0.000078      185     0.004662    terminé


     85900     0.044200     0.000125      185     0.005468    terminé


Facteur de temps DP mesuré : 99.46 ; capacité multipliée par 100.


### Lecture du résultat

Les poids et la capacité sont multipliés ensemble par 1, 10 puis 100 : le problème combinatoire reste identique. La DP parcourt une capacité plus grande tandis que le B&B explore les mêmes décisions. Les durées affichées dépendent de la machine ; ni leur monotonie ni un facteur précis ne sont garantis.

### 3.3 Ce qui fait exploser le B&B : la correlation, pas la taille

Le second critere est l'inverse. A `n` et **ratio de capacite** fixes, on fait varier la **famille d'instance** et on mesure deux choses : l'ecart relatif entre la borne LP a la racine et l'optimum (qualite de la borne), et le nombre de nœuds developpes (ce que la borne a effectivement permis d'elaguer).


In [9]:
var famillesResultats=new List<(string nom,int optimum,double borne,double ecart,int noeuds,double temps,double glouton,bool epuise)>();
Console.WriteLine($"{"famille",-22} {"optimum",8} {"borne LP",10} {"écart",9} {"nœuds",8} {"B&B (s)",10} {"glouton",9} {"budget",8}");
foreach(string f in Familles){var x=Genere(40,0.5,f,42);int opt=MILP(x).valeur;double b=Borne(x,Ordre(x),x.Capacite);if(b+1e-8<opt)throw new Exception("Borne inadmissible");var sw=Stopwatch.StartNew();var bb=BB(x);sw.Stop();int g=Glouton(x);if(!bb.epuise&&bb.valeur!=opt)throw new Exception("B&B incorrect");double gap=(b-opt)/opt,eg=(double)(opt-g)/opt;famillesResultats.Add((f,opt,b,gap,bb.noeuds,sw.Elapsed.TotalSeconds,eg,bb.epuise));Console.WriteLine($"{f,-22} {opt,8} {b,10:F2} {gap,9:P2} {bb.noeuds,8} {sw.Elapsed.TotalSeconds,10:F6} {eg,9:P2} {(bb.epuise?"épuisé":"terminé"),8}");}

famille                 optimum   borne LP     écart    nœuds    B&B (s)   glouton   budget


non_correlee               1199    1206.10    0.59 %      185   0.000069    1.67 %  terminé


faiblement_correlee         908     910.87    0.32 %      248   0.000073    0.33 %  terminé


fortement_correlee         1139    1147.14    0.71 %    21394   0.004291    4.21 %  terminé


### Lecture du résultat

Le tableau donne pour chaque famille l’optimum SCIP, la borne fractionnaire, leur écart relatif, les nœuds, le temps et le déficit du glouton. Une borne serrée à la racine ne garantit pas une petite recherche : c’est aussi sa capacité à discriminer les sous-problèmes qui compte. Les rapports entre familles sont propres à ces instances.

### Exercice 2 — ordre de branchement

Affectez `ordrePoids` à une fonction renvoyant les indices par poids croissant, puis par indice en cas d’égalité. Le harnais effectue le DFS avec cet ordre, trie les objets restants par ratio pour la borne fractionnaire et compte lui-même les nœuds. Comparez avec le B&B initial.

In [10]:
Func<Instance,int[]> ordrePoids=null; // TODO étudiant : indices des objets par poids croissant (égalité : indice croissant).
if(ordrePoids is null)Console.WriteLine("Exercice à compléter : construire l'ordre de branchement par poids.");
else {foreach(string f in Familles){var x=Genere(12,0.5,f,42);int[] ordre=ordrePoids(x);var attendu=Enumerable.Range(0,x.Poids.Length).OrderBy(i=>x.Poids[i]).ThenBy(i=>i).ToArray();if(ordre is null||!ordre.SequenceEqual(attendu))throw new Exception("Ordre des poids invalide");int meilleur=0,noeuds=0;void Explore(int i,int reste,int valeur){noeuds++;meilleur=Math.Max(meilleur,valeur);if(i==ordre.Length)return;var resteTrie=ordre.Skip(i).OrderByDescending(j=>(double)x.Valeurs[j]/x.Poids[j]).ThenBy(j=>j);if(valeur+Borne(x,resteTrie,reste)<=meilleur+1e-10)return;int j=ordre[i];if(x.Poids[j]<=reste)Explore(i+1,reste-x.Poids[j],valeur+x.Valeurs[j]);Explore(i+1,reste,valeur);}Explore(0,x.Capacite,0);if(meilleur!=DP(x))throw new Exception("Optimum invalide");Console.WriteLine($"{f}: optimum={meilleur}, nœuds réellement visités={noeuds}; ordre ratio={BB(x).noeuds}");}}

Exercice à compléter : construire l'ordre de branchement par poids.


### Figure — deux axes de coût

Les deux panneaux représentent les nœuds par famille et les temps selon la capacité sur des échelles logarithmiques.

In [11]:
record SvgBB(string Content);
Formatter.Register<SvgBB>((v,w)=>w.Write(v.Content),"text/html");string F(double x)=>x.ToString("0.###",CultureInfo.InvariantCulture);
var sb=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='1000' height='430' viewBox='0 0 1000 430'><rect width='100%' height='100%' fill='white'/><text x='250' y='25' text-anchor='middle'>Nœuds B&amp;B par famille</text><text x='750' y='25' text-anchor='middle'>Temps selon la capacité</text><path d='M80 50 V330 H450 M570 50 V330 H950' stroke='black' fill='none'/>");
double maxN=Math.Ceiling(Math.Log10(famillesResultats.Max(r=>(double)r.noeuds)));maxN=Math.Max(1,maxN);
for(int k=0;k<=maxN;k++){double y=330-k*280/maxN;sb.Append($"<path d='M80 {F(y)} H450' stroke='#ddd'/><text x='75' y='{F(y+4)}' text-anchor='end'>10^{k}</text>");}
string[] noms={"non corrélée","faiblement","fortement"};string[] couleurs={"#4c72b0","#dd8452","#c44e52"};for(int i=0;i<3;i++){double y=330-Math.Log10(famillesResultats[i].noeuds)*280/maxN;sb.Append($"<rect x='{115+i*110}' y='{F(y)}' width='60' height='{F(330-y)}' fill='{couleurs[i]}'/><text x='{145+i*110}' y='350' text-anchor='middle' font-size='12'>{noms[i]}</text>");}
double lo=Math.Floor(Math.Log10(capacites.Min(z=>Math.Min(z.dp,z.bb)))),hi=Math.Ceiling(Math.Log10(capacites.Max(z=>Math.Max(z.dp,z.bb))));if(hi<=lo)hi=lo+1;double X(int i)=>590+i*170;double Y(double v)=>330-(Math.Log10(v)-lo)*280/(hi-lo);
for(int k=(int)lo;k<=hi;k++){double y=330-(k-lo)*280/(hi-lo);sb.Append($"<path d='M570 {F(y)} H950' stroke='#ddd'/><text x='565' y='{F(y+4)}' text-anchor='end'>10^{k}</text>");}
foreach(var (serie,col) in new[]{(capacites.Select(z=>z.dp).ToArray(),"#0645ad"),(capacites.Select(z=>z.bb).ToArray(),"#a00000")}){sb.Append($"<polyline fill='none' stroke='{col}' stroke-width='2' points='{string.Join(" ",serie.Select((v,i)=>F(X(i))+","+F(Y(v))))}'/>");foreach(var (v,i) in serie.Select((v,i)=>(v,i)))sb.Append($"<circle cx='{F(X(i))}' cy='{F(Y(v))}' r='4' fill='{col}'/>");}
for(int i=0;i<3;i++)sb.Append($"<text x='{F(X(i))}' y='350' text-anchor='middle'>{capacites[i].W}</text>");sb.Append("<text x='250' y='390' text-anchor='middle'>famille (n = 40)</text><text x='750' y='390' text-anchor='middle'>capacité W (échelle log)</text><text x='18' y='200' transform='rotate(-90 18 200)' text-anchor='middle'>nœuds (échelle log)</text><text x='507' y='200' transform='rotate(-90 507 200)' text-anchor='middle'>temps en s (échelle log)</text><text x='600' y='45' fill='#0645ad'>DP</text><text x='660' y='45' fill='#a00000'>B&amp;B</text></svg>");display(new SvgBB(sb.ToString()));Console.WriteLine("Deux panneaux : nœuds par famille et temps DP/B&B selon la capacité, axes logarithmiques.");

Nœuds B&B par famille Temps selon la capacité 10^0 10^1 10^2 10^3 10^4 10^5 non corrélée faiblement fortement 10^-5 10^-4 10^-3 10^-2 10^-1 859 8590 85900 famille (n = 40) capacité W (échelle log) nœuds (échelle log) temps en s (échelle log) DP B&B

Deux panneaux : nœuds par famille et temps DP/B&B selon la capacité, axes logarithmiques.


### Lecture de la figure

À gauche, les nœuds visités sont comparés entre familles à 40 objets ; à droite, les temps DP et B&B sont mesurés quand la capacité varie. Les axes logarithmiques facilitent la lecture des écarts sans garantir de domination universelle.

### Exercice 3 — frontière mesurée

Affectez `frontiere` à une fonction recevant les capacités et les deux tableaux de temps mesurés. Elle renvoie la première capacité pour laquelle B&B est plus rapide, ou `null` si aucune ne l’est. Le harnais appelle lui-même DP et B&B, mesure leurs temps et contrôle la réponse. Le bruit rend la frontière indicative.

Tracez les frontières obtenues pour les trois familles à partir de ces appels mesurés ; discutez leur ordre et signalez les familles sans croisement observé. Le graphique et son interprétation sont à réaliser par l’étudiant.

In [12]:
Func<int[],double[],double[],int?> frontiere=null; // TODO étudiant : choisir la première capacité où B&B est plus rapide.
if(frontiere is null)Console.WriteLine("Exercice à compléter : déterminer la frontière à partir des mesures.");
else {foreach(string f in Familles){var x=Genere(40,0.5,f,42);int[] ws={x.Capacite/4,x.Capacite/2,x.Capacite};var ds=new double[3];var bs=new double[3];for(int i=0;i<3;i++){var y=x with {Capacite=ws[i]};var sw=Stopwatch.StartNew();int d=DP(y);sw.Stop();ds[i]=sw.Elapsed.TotalSeconds;sw.Restart();var b=BB(y);sw.Stop();bs[i]=sw.Elapsed.TotalSeconds;if(!b.epuise&&b.valeur!=d)throw new Exception("Désaccord");Console.WriteLine($"{f}, W={ws[i]} : DP={ds[i]:E3}s B&B={bs[i]:E3}s, nœuds={b.noeuds}, épuisé={b.epuise}");}int? attendu=Enumerable.Range(0,3).Where(i=>bs[i]<ds[i]).Select(i=>(int?)ws[i]).FirstOrDefault();int? obtenu=frontiere(ws,ds,bs);if(obtenu!=attendu)throw new Exception("Frontière incorrecte pour ces mesures");Console.WriteLine($"Frontière observée : {obtenu?.ToString() ?? "aucune"}");}}

Exercice à compléter : déterminer la frontière à partir des mesures.


## 4. Synthèse : quel algorithme exact choisir

| Situation | Méthode | Critère |
|---|---|---|
| Capacité modérée | DP | Coût pseudo-polynomial $O(nW)$. |
| Bonne discrimination | B&B | L’élagage peut réduire l’arbre ; surveiller le budget. |
| Formulation générale | SCIP | Solveur MILP, témoin et borne vérifiés. |
| Solution rapide sans garantie | Glouton | Comparer sa valeur à un optimum ou une borne. |

Le coût dépend de la taille, de la capacité et de la structure : une seule variable ne suffit pas à caractériser la difficulté.

## 5. Exercices

Trois exercices répartis dans le parcours : [reconstruction DP](#exercice-1--reconstruire-la-solution), [ordre de branchement](#exercice-2--ordre-de-branchement) et [frontière mesurée](#exercice-3--frontière-mesurée). Les stubs restent à compléter.

## À retenir

- La DP coûte $O(nW)$ : pseudo-polynomiale en capacité.
- Le B&B combine recherche et majorant fractionnaire ; le compteur et l’état du budget sont observables.
- SCIP résout le MILP et fournit une borne et un témoin ; 9 instances vérifient l’accord avec les méthodes maison.
- La qualité à la racine ne suffit pas à prédire la taille de l’arbre ; les chronos dépendent de l’environnement.

## Vérifiez votre compréhension

1. Pourquoi la DP est-elle pseudo-polynomiale ?
2. Pourquoi une relaxation fractionnaire donne-t-elle une borne supérieure ?
3. Pourquoi le nombre de nœuds peut-il varier à taille fixe ?

<details><summary>Réponses</summary>

1. Sa complexité dépend de la valeur numérique $W$, pas seulement de sa longueur binaire.
2. Elle autorise toutes les solutions entières et davantage de solutions.
3. La structure des rapports et le pouvoir d’élagage changent entre instances.

</details>

## Pour aller plus loin

- [Search-09 — Programmation linéaire](Search-09-LinearProgramming.ipynb) : formulation du sac à dos.
- [Search-11d — Descente sous budget](Search-11d-Descente-Sous-Budget.ipynb) : borne d’un voisinage.
- [Search-03b — Pattern databases](Search-03b-PatternDatabases.ipynb) : borne admissible en recherche.

**Navigation** : [Search-11d](Search-11d-Descente-Sous-Budget.ipynb) | [Index](../README.md) | [Version Python](Search-12-BranchAndBound-Borne-LP-Python.ipynb) | [Search-12a](Search-12a-Composer-Regards.ipynb)